# ColdStart Killer — Build 3K MVP Dataset from Amazon Reviews 2023

⚠️ Run this notebook only when you are ready to stream/download the selected Amazon Reviews 2023 metadata files.

This notebook:
1. Loads selected Amazon Reviews 2023 metadata.
2. Normalizes product fields.
3. Preserves product images.
4. Builds a high-quality, category-diverse 3,000-item MVP dataset.
5. Saves CSV and reports into `analysis/`.

Pipeline output: the main handoff file is `analysis/mvp_3000_items_diverse.csv`. It contains canonical product metadata, parsed price/image fields, category labels, quality scores, and `product_text_for_llm`, which Notebook 02 uses as grounded evidence for proposition and HyPE generation.

## 1. Setup imports and paths

Load project modules, resolve the repository root, and import helper functions used for dataset loading, normalization, selection, validation, and output writing.

In [2]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.dataset_loading import load_target_metadata
from src.normalize_amazon import normalize_record
from src.mvp_selection import (
    add_scoring_columns,
    build_quality_control_subset,
    build_diverse_subset,
    category_summary,
    write_mvp_outputs,
)
from src.validation import validate_mvp_dataframe

## 2. Load metadata

Load raw Amazon Reviews 2023 metadata from the configured source categories. This is the raw input pool only: it collects records plus a load report, while text-rich filtering and category balancing happen in later cells.

In [3]:
records, load_report = load_target_metadata()
load_report

All_Beauty parquet: 112590it [00:09, 12116.99it/s]
Cell_Phones stream sample: 100%|██████████| 20000/20000 [00:10<00:00, 1835.72it/s]


{'sources': {'all_beauty': 'All_Beauty_parquet',
  'cell_phones': 'Cell_Phones_jsonl'},
 'row_counts': {'all_beauty': 112590,
  'cell_phones': 20000,
  'combined': 132590},
 'failures': [],
 'policy': {'cell_sample_n': 20000,
  'review_sample_optional': True,
  'reviews_used_for_retrieval_features': False}}

## 3. Normalize metadata

Convert raw records into a consistent product table. This step cleans title/features/description/details, parses price and images, builds `product_text_for_llm` as the canonical LLM evidence text, and adds quality scoring columns.

In [4]:
normalized_rows = [
    normalize_record(record, category_key=record.get("_source_category", "unknown"))
    for record in records
]
df = pd.DataFrame(normalized_rows)
df = add_scoring_columns(df)
df.shape

(132590, 50)

## 4. Show field coverage

Measure how often important product fields are present after normalization. Use this quick coverage check to understand source-data quality before selecting the MVP subset.

In [5]:
coverage = {
    "title": df["title_present"].mean(),
    "store": df["store_present"].mean(),
    "categories": df["categories_present"].mean(),
    "price": df["price_usd"].notna().mean(),
    "details": df["details_present"].mean(),
}
pd.Series(coverage).sort_values(ascending=False)

categories    1.000000
title         0.999909
details       0.965782
store         0.913093
price         0.188415
dtype: float64

## 5. Show description vs combined text coverage

Inspect text richness from description, features, details, and the combined text field. `combined_words` is the main richness signal: products need enough grounded seller/metadata text before Qwen extracts propositions and HyPE buyer queries.

In [6]:
df[["description_words", "features_words", "details_words", "combined_words"]].describe()

,description_words,features_words,details_words,combined_words
count,132590.000000,132590.000000,132590.000000,132590.000000
mean,20.360495,24.668437,32.032122,77.061053
std,59.263813,59.617330,23.584061,111.399694
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,19.000000,22.000000
50%,0.000000,0.000000,30.000000,33.000000
75%,0.000000,0.000000,38.000000,61.000000
max,2437.000000,679.000000,872.000000,2580.000000


## 6. Show image coverage

Check how many products have usable image URLs. Images are not required for retrieval, but strong image coverage makes the demo and UI outputs more useful.

In [7]:
image_coverage = {
    "has_images": df["has_images"].mean(),
    "primary_image_url": df["primary_image_url"].notna().mean(),
    "avg_images_count": df["images_count"].mean(),
}
image_coverage

{'has_images': np.float64(0.9999321215777962),
 'primary_image_url': np.float64(0.9999321215777962),
 'avg_images_count': np.float64(14.26504261256505)}

## 7. Build quality-control 3k dataset

Create a strict top-quality reference subset using richness filters and quality ranking. This is a quality-control comparison file, useful for auditing whether the filter can find rich products, but it is not the default MongoDB insert input.

In [8]:
control_df = build_quality_control_subset(df, target_n=3000)
{
    "rows": len(control_df),
    "unique_parent_asin": control_df["parent_asin"].nunique(),
    "selected_threshold": control_df.attrs.get("selected_threshold"),
    "warnings": control_df.attrs.get("warnings", []),
}

{'rows': 3000,
 'unique_parent_asin': 3000,
 'selected_threshold': 150,
 'warnings': []}

## 8. Build diverse 3k dataset

Create the primary MVP dataset for MongoDB insertion. This selection keeps text-rich, unique products while capping dominant categories, so the final 3,000 rows become a more diverse cold-start catalog.

In [9]:
diverse_df = build_diverse_subset(df, target_n=3000, min_combined_words=150, max_per_category=1200)
{
    "rows": len(diverse_df),
    "unique_parent_asin": diverse_df["parent_asin"].nunique(),
    "selected_threshold": diverse_df.attrs.get("selected_threshold"),
    "warnings": diverse_df.attrs.get("warnings", []),
}

{'rows': 3000,
 'unique_parent_asin': 3000,
 'selected_threshold': 150,
 'warnings': []}

## 9. Show category summary

Summarize the selected diverse dataset by category. This verifies category balance, average text richness, quality score, and image coverage before writing files.

In [10]:
pd.DataFrame(category_summary(diverse_df)).head(25)

,category_id,main_category,rows,avg_quality_score,avg_combined_words,image_coverage
0,cell_phones_and_accessories,Cell Phones & Accessories,1200,1.000000,460.77,1.0
1,all_beauty,All Beauty,803,1.000000,431.88,1.0
2,amazon_fashion,AMAZON FASHION,436,0.790521,270.81,1.0
3,all_electronics,All Electronics,381,0.853731,399.03,1.0
4,camera_and_photo,Camera & Photo,39,0.835590,363.23,1.0
5,computers,Computers,30,0.881150,336.43,1.0
6,industrial_and_scientific,Industrial & Scientific,25,0.871053,465.24,1.0
7,sports_and_outdoors,Sports & Outdoors,25,0.858353,283.68,1.0
8,amazon_home,Amazon Home,16,0.865177,367.62,1.0
9,portable_audio_and_accessories,Portable Audio & Accessories,14,0.963048,363.14,1.0


## 11. Save CSVs and reports

Write the final CSVs and audit reports into `analysis/`. The key downstream file is `analysis/mvp_3000_items_diverse.csv`: Notebook 02 reads it and turns each row into one `items` document plus multiple `retrieval_units`.

In [11]:
write_mvp_outputs(df, control_df, diverse_df, load_report, output_dir=ROOT / "analysis")
sorted(str(path.relative_to(ROOT)) for path in (ROOT / "analysis").glob("mvp_3000*"))

['analysis\\mvp_3000_items.csv',
 'analysis\\mvp_3000_items_diverse.csv',
 'analysis\\mvp_3000_selection_report.json']

## 12. Final checks

Validate row count, product uniqueness, category count, text threshold, image coverage, price coverage, and required columns before moving to MongoDB indexing. If this check is clean, the CSV is ready to become the indexing pipeline input.

In [12]:
selected_threshold = diverse_df.attrs.get("selected_threshold", 150)
checks = {
    "row_count": len(diverse_df),
    "unique_parent_asin": diverse_df["parent_asin"].nunique(),
    "category_count": diverse_df["category_id"].nunique(),
    "avg_combined_words": diverse_df["combined_words"].mean(),
    "image_coverage": diverse_df["primary_image_url"].notna().mean(),
    "price_coverage": diverse_df["price_usd"].notna().mean(),
    "validation": validate_mvp_dataframe(diverse_df, min_combined_words=selected_threshold),
}
checks

{'row_count': 3000,
 'unique_parent_asin': 3000,
 'category_count': 20,
 'avg_combined_words': np.float64(410.9273333333333),
 'image_coverage': np.float64(1.0),
 'price_coverage': np.float64(1.0),
 'validation': {'ok': True,
  'errors': [],
  'warnings': [],
  'image_coverage': 1.0}}